# UFC — Golpes significativos e decisões
**Igor de Andrade Albuquerque · Fase 2 · BI e Analytics**

Este notebook reproduz a coleta versionada, limpeza, banco SQLite, KPIs, modelo e painéis. No Colab, use **Ambiente de execução → Executar tudo**. Não exige GPU nem montagem do Drive. A primeira execução baixa três CSVs públicos (aproximadamente 10 MB).

**Pergunta de negócio:** com que frequência o líder em golpes significativos vence a decisão e como isso muda entre unânimes e divididas? O usuário é um analista de cobertura esportiva. O uso proposto é conferir afirmações e escolher casos para análise editorial.

**Escopo:** UFC, 2022–2024, formato previsto 3 × 5 minutos, três rounds completos, decisões unânimes ou divididas e vencedor identificável. Os empates na contagem permanecem nos KPIs e ficam fora do classificador.

**Limites:** os indicadores não medem erro ou viés dos juízes. A previsão usa estatísticas de combates concluídos; não é pré-luta. A conferência externa das 12 fichas da amostra ficou pendente por bloqueio de acesso; os testes internos foram concluídos.

## 1. Preparação do ambiente
A célula cria uma pasta isolada quando os arquivos do repositório não estão presentes. As fontes do projeto estão incorporadas para permitir executar este notebook sozinho. Dependências: pandas, numpy, matplotlib, scikit-learn e IPython (incluídas no Colab).

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'data/raw/ufc_event_details.csv').exists():
    ROOT = (Path('/content/ufc_bi') if Path('/content').exists() else ROOT / 'ufc_bi_runtime')
for folder in ['src','data/raw','data/processed','outputs','tests','docs','notebooks']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
SOURCES = {'src/source_check.py': '"""Registra acessibilidade da amostra oficial; HTTP 200 não valida campos."""\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import datetime,timezone\nfrom urllib.request import Request,urlopen\nimport json\nimport pandas as pd\nfrom .pipeline import ROOT\n\ndef check(row):\n    record=dict(row,checked_at=datetime.now(timezone.utc).isoformat(),field_validation=\'pendente\')\n    try:\n        req=Request(row[\'url\'],headers={\'User-Agent\':\'Mozilla/5.0 (academic data verification)\'})\n        with urlopen(req,timeout=12) as response:\n            body=response.read()\n            (ROOT/\'outputs/source_pages\').mkdir(exist_ok=True)\n            (ROOT/f"outputs/source_pages/{row[\'bout_id\']}.html").write_bytes(body)\n            valid_page=b\'b-fight-details__table\' in body\n            record.update(http_status=response.status,access=\'disponível; conferir campos\' if valid_page else \'bloqueio de acesso; resposta sem tabela da luta\')\n    except Exception as e:record.update(http_status=None,access=\'indisponível\',error=str(e))\n    return record\n\nif __name__==\'__main__\':\n    sample=pd.read_csv(ROOT/\'data/processed/source_check_sample.csv\').to_dict(\'records\')\n    with ThreadPoolExecutor(max_workers=4) as pool:results=list(pool.map(check,sample))\n    (ROOT/\'outputs/source_checks.json\').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print([(r[\'bout_id\'],r[\'access\']) for r in results])\n', 'src/pipeline.py': '"""Coleta versionada, qualidade, modelo relacional e KPIs do UFC.\n\nExecute da raiz: python -m src.pipeline. Nenhum dado sintético integra a análise.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport re\nimport sqlite3\nfrom urllib.request import urlopen\nimport numpy as np\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nCOMMIT = \'c742842f29466758fedc3906dbad28d6d320a8da\'\nHASHES = {\n \'ufc_event_details.csv\': \'07355c6e414bb5932a7a0f97e4088388ff7391876b7a9ab792433c83f5285841\',\n \'ufc_fight_results.csv\': \'b99d94a82984f07ec34d9f04423fd7edd67ea28c365621a144e05aec64f3a840\',\n \'ufc_fight_stats.csv\': \'f5b316d5e745bd89da921b2c9711ec9209c2fa39dcbcd80bcb063b62d9175bcb\'}\n\ndef collect():\n    """Baixa somente arquivos ausentes e exige os hashes do snapshot da Fase 1."""\n    raw = ROOT/\'data/raw\'; raw.mkdir(parents=True, exist_ok=True)\n    manifest=[]\n    for name, expected in HASHES.items():\n        path=raw/name\n        url=f\'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/{COMMIT}/{name}\'\n        if not path.exists():\n            with urlopen(url, timeout=90) as response: payload=response.read()\n            if hashlib.sha256(payload).hexdigest()!=expected: raise ValueError(f\'Hash inesperado: {name}\')\n            path.write_bytes(payload)\n        digest=hashlib.sha256(path.read_bytes()).hexdigest()\n        if digest!=expected: raise ValueError(f\'Arquivo alterado: {name}\')\n        manifest.append(dict(file=name,bytes=path.stat().st_size,sha256=digest,url=url))\n    (ROOT/\'outputs\').mkdir(exist_ok=True)\n    (ROOT/\'outputs/manifest.json\').write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\n    return {n:pd.read_csv(raw/n,dtype=str,keep_default_na=False).apply(lambda c:c.str.strip()) for n in HASHES}\n\ndef pair(value):\n    """Converte \'acertos of tentativas\'; vazio nunca vira zero."""\n    match=re.fullmatch(r\'(\\d+) of (\\d+)\',str(value))\n    if not match: raise ValueError(\'par ausente ou malformado\')\n    landed,attempted=map(int,match.groups())\n    if landed>attempted: raise ValueError(\'acertos acima das tentativas\')\n    return landed,attempted\n\ndef outcome_winner(bout,outcome):\n    names=bout.split(\' vs. \')\n    if len(names)!=2 or outcome not in [\'W/L\',\'L/W\']: raise ValueError(\'vencedor ambíguo\')\n    return names[0 if outcome==\'W/L\' else 1]\n\ndef classify(a,b,winner):\n    if a[\'sig_landed\']==b[\'sig_landed\']: return \'E\'\n    leader=a if a[\'sig_landed\']>b[\'sig_landed\'] else b\n    return \'C\' if leader[\'fighter\']==winner else \'D\'\n\ndef kpis(frame):\n    c=int((frame.status==\'C\').sum());d=int((frame.status==\'D\').sum());e=int((frame.status==\'E\').sum());n=c+d+e\n    return dict(n=n,C=c,D=d,E=e,concordance=100*c/(c+d) if c+d else None,\n                divergence=100*d/(c+d) if c+d else None,coverage=100*(c+d)/n if n else None,\n                sensitivity_low=100*c/n if n else None,sensitivity_high=100*(c+e)/n if n else None)\n\ndef optional_int(value):\n    try:\n        x=float(value)\n        return int(x) if x>=0 and x.is_integer() else np.nan\n    except ValueError: return np.nan\n\ndef control_seconds(value):\n    match=re.fullmatch(r\'(\\d+):([0-5]\\d)\',value)\n    if not match:return np.nan\n    m,s=map(int,match.groups());return 60*m+s if 60*m+s<=300 else np.nan\n\ndef build():\n    tables=collect();ev=tables[\'ufc_event_details.csv\'];res=tables[\'ufc_fight_results.csv\'];st=tables[\'ufc_fight_stats.csv\']\n    audit={\'raw_rows\':{k:len(v) for k,v in tables.items()}}\n    ev[\'date\']=pd.to_datetime(ev.DATE,format=\'%B %d, %Y\',errors=\'coerce\')\n    if ev.EVENT.duplicated().any(): raise ValueError(\'Evento ambíguo no catálogo\')\n    res=res.merge(ev[[\'EVENT\',\'date\',\'LOCATION\']],on=\'EVENT\',how=\'left\',validate=\'many_to_one\')\n    audit[\'result_rows_without_date_before_alias\']=int(res.date.isna().sum())\n    # A URL concilia nomes alternativos. Uma URL não pode ter duas datas conhecidas.\n    if res.groupby(\'URL\').date.nunique().gt(1).any():raise ValueError(\'Datas conflitantes para a mesma URL\')\n    res[\'date\']=res.groupby(\'URL\').date.transform(\'max\')\n    audit[\'result_rows_without_date_after_alias\']=int(res.date.isna().sum())\n    p=ROOT/\'data/processed\';p.mkdir(exist_ok=True)\n    res[res.date.isna()][[\'EVENT\',\'BOUT\',\'URL\']].to_csv(p/\'unresolved_dates.csv\',index=False)\n    in_period=res[res.date.dt.year.between(2022,2024)].copy()\n    audit[\'bouts_in_period\']=int(in_period.URL.nunique())\n    mask=(in_period[\'TIME FORMAT\']==\'3 Rnd (5-5-5)\')&(in_period.ROUND==\'3\')&(in_period.TIME==\'5:00\')&in_period.METHOD.isin([\'Decision - Unanimous\',\'Decision - Split\'])&in_period.OUTCOME.isin([\'W/L\',\'L/W\'])\n    candidates=in_period[mask].copy();audit[\'eligible_result_rows\']=len(candidates)\n    # Consistência dos resultados duplicados antes de remover aliases.\n    for col in [\'BOUT\',\'OUTCOME\',\'METHOD\',\'ROUND\',\'TIME\',\'TIME FORMAT\',\'WEIGHTCLASS\']:\n        if candidates.groupby(\'URL\')[col].nunique().gt(1).any():raise ValueError(f\'Resultado conflitante: {col}\')\n    mapping=candidates[[\'EVENT\',\'BOUT\',\'URL\']].drop_duplicates()\n    if mapping.duplicated([\'EVENT\',\'BOUT\']).any():raise ValueError(\'Chave textual ambígua no recorte\')\n    candidates=candidates.sort_values([\'URL\',\'EVENT\']).drop_duplicates(\'URL\')\n    audit[\'eligible_bouts\']=len(candidates)\n    joined=st.merge(mapping,on=[\'EVENT\',\'BOUT\'],how=\'inner\',validate=\'many_to_one\')\n    audit[\'round_rows_before_dedup\']=len(joined)\n    keys=[\'URL\',\'FIGHTER\',\'ROUND\'];vals=[c for c in st.columns if c not in [\'EVENT\',\'BOUT\']+keys]\n    conflict=joined.groupby(keys)[vals].nunique().max(axis=1).gt(1)\n    bad_urls=set(conflict[conflict].index.get_level_values(0))\n    joined=joined.drop_duplicates(keys)\n    audit[\'round_rows_after_dedup\']=len(joined)\n    groups={url:g for url,g in joined.groupby(\'URL\')}\n    exclusions=[];rounds=[];bouts=[]\n    for _,r in candidates.iterrows():\n        try:\n            if r.URL in bad_urls:raise ValueError(\'duplicatas conflitantes\')\n            names=r.BOUT.split(\' vs. \');g=groups.get(r.URL,pd.DataFrame())\n            expected={(f,f\'Round {i}\') for f in names for i in (1,2,3)}\n            if len(names)!=2 or len(g)!=6 or set(zip(g.FIGHTER,g.ROUND))!=expected:raise ValueError(\'grade de atletas/rounds incompleta\')\n            rr=[]\n            for _,s in g.iterrows():\n                landed,attempted=pair(s[\'SIG.STR.\'])\n                try:td,_=pair(s[\'TD\'])\n                except ValueError:td=np.nan\n                rr.append(dict(bout_id=r.URL.rsplit(\'/\',1)[-1],fighter=s.FIGHTER,round=int(s.ROUND[-1]),sig_landed=landed,sig_attempted=attempted,\n                               td_landed=td,kd=optional_int(s.KD),control_seconds=control_seconds(s.CTRL)))\n            total=pd.DataFrame(rr).groupby(\'fighter\')[[\'sig_landed\',\'sig_attempted\',\'td_landed\',\'kd\',\'control_seconds\']].agg(lambda x:x.sum(min_count=len(x)))\n            totals=[dict(fighter=f,**total.loc[f].to_dict()) for f in names];winner=outcome_winner(r.BOUT,r.OUTCOME)\n            status=classify(*totals,winner)\n            a,b=sorted(totals,key=lambda x:(-x[\'sig_landed\'],x[\'fighter\']))\n            ar=[x for x in rr if x[\'fighter\']==a[\'fighter\']];br=[x for x in rr if x[\'fighter\']==b[\'fighter\']]\n            ar=sorted(ar,key=lambda x:x[\'round\']);br=sorted(br,key=lambda x:x[\'round\'])\n            row=dict(bout_id=rr[0][\'bout_id\'],date=r.date.strftime(\'%Y-%m-%d\'),year=int(r.date.year),event=r.EVENT,\n                     method=\'Unânime\' if r.METHOD.endswith(\'Unanimous\') else \'Dividida\',weightclass=r.WEIGHTCLASS,\n                     fighter_a=names[0],fighter_b=names[1],winner=winner,status=status,\n                     leader=a[\'fighter\'] if status!=\'E\' else \'\',other=b[\'fighter\'] if status!=\'E\' else \'\',\n                     sig_a=int(totals[0][\'sig_landed\']),sig_b=int(totals[1][\'sig_landed\']),url=r.URL)\n            row[\'sig_margin\']=a[\'sig_landed\']-b[\'sig_landed\']\n            row[\'attempt_margin\']=a[\'sig_attempted\']-b[\'sig_attempted\']\n            row[\'accuracy_margin\']=(a[\'sig_landed\']/a[\'sig_attempted\'] if a[\'sig_attempted\'] else np.nan)-(b[\'sig_landed\']/b[\'sig_attempted\'] if b[\'sig_attempted\'] else np.nan)\n            row[\'round_margin\']=sum(int(x[\'sig_landed\']>y[\'sig_landed\'])-int(x[\'sig_landed\']<y[\'sig_landed\']) for x,y in zip(ar,br))\n            for col in [\'td_landed\',\'kd\',\'control_seconds\']:row[col+\'_margin\']=a[col]-b[col]\n            bouts.append(row);rounds.extend(rr)\n        except (ValueError,AttributeError) as error: exclusions.append(dict(url=r.URL,reason=str(error)))\n    b=pd.DataFrame(bouts).sort_values([\'date\',\'bout_id\']).reset_index(drop=True);rd=pd.DataFrame(rounds)\n    b.to_csv(p/\'bouts.csv\',index=False);rd.to_csv(p/\'rounds.csv\',index=False)\n    pd.DataFrame(exclusions,columns=[\'url\',\'reason\']).to_csv(p/\'exclusions.csv\',index=False)\n    audit.update(valid_bouts=len(b),excluded_bouts=len(exclusions),valid_round_rows=len(rd),optional_missing=rd.isna().sum().to_dict())\n    summary={\'all\':kpis(b),\'by_method\':{str(m):kpis(g) for m,g in b.groupby(\'method\')},\'by_year\':{str(y):kpis(g) for y,g in b.groupby(\'year\')},\n             \'by_year_method\':{f\'{y} {m}\':kpis(g) for (y,m),g in b.groupby([\'year\',\'method\'])}}\n    summary[\'H1_excess_pp\']=summary[\'all\'][\'concordance\']-50\n    summary[\'H2_difference_pp\']=summary[\'by_method\'][\'Unânime\'][\'concordance\']-summary[\'by_method\'][\'Dividida\'][\'concordance\']\n    for name,obj in [(\'quality\',audit),(\'kpis\',summary)]:\n        (ROOT/f\'outputs/{name}.json\').write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    # Modelo normalizado: resultados por luta, dimensão atleta e estatísticas por atleta/round.\n    db=p/\'ufc.sqlite\'\n    if db.exists():db.unlink()\n    with sqlite3.connect(db) as conn:\n        conn.execute(\'PRAGMA foreign_keys=ON\')\n        names=sorted(set(rd.fighter));ath=pd.DataFrame({\'fighter_id\':range(1,len(names)+1),\'fighter\':names})\n        ath.to_sql(\'dim_fighter\',conn,index=False);conn.execute(\'CREATE UNIQUE INDEX ux_fighter ON dim_fighter(fighter_id)\')\n        b.to_sql(\'fact_bout\',conn,index=False);conn.execute(\'CREATE UNIQUE INDEX ux_bout ON fact_bout(bout_id)\')\n        rd.merge(ath,on=\'fighter\',validate=\'many_to_one\').drop(columns=\'fighter\').to_sql(\'fact_round\',conn,index=False)\n        conn.execute(\'CREATE UNIQUE INDEX ux_round ON fact_round(bout_id,fighter_id,round)\')\n        conn.execute(\'CREATE VIEW vw_bout_kpis AS SELECT year,method,COUNT(*) AS n,SUM(status="C") AS concordances,SUM(status="D") AS divergences,SUM(status="E") AS ties,100.0*SUM(status="C")/NULLIF(SUM(status IN ("C","D")),0) AS concordance_pct FROM fact_bout GROUP BY year,method\')\n        check=pd.read_sql(\'SELECT SUM(n) n FROM vw_bout_kpis\',conn).iloc[0,0]\n        assert int(check)==len(b)\n    sample=b.groupby([\'year\',\'method\'],group_keys=False).apply(lambda g:g.sort_values(\'url\').head(2),include_groups=False).reset_index()\n    sample[[\'bout_id\',\'url\']].to_csv(p/\'source_check_sample.csv\',index=False)\n    print(json.dumps({\'quality\':audit,\'kpis\':summary},ensure_ascii=False,indent=2))\n    return b,rd\n\nif __name__==\'__main__\':build()\n', 'src/visuals.py': '"""Gera gráficos documentais e o dashboard interativo incorporável ao Colab."""\nimport json\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom .pipeline import ROOT\n\ndef build():\n    out=ROOT/\'outputs\';b=pd.read_csv(ROOT/\'data/processed/bouts.csv\');r=pd.read_csv(ROOT/\'data/processed/rounds.csv\');p=pd.read_csv(out/\'predictions_2024.csv\')\n    k=json.loads((out/\'kpis.json\').read_text());m=json.loads((out/\'model_metrics.json\').read_text());q=json.loads((out/\'quality.json\').read_text())\n    checks=json.loads((out/\'source_checks.json\').read_text()) if (out/\'source_checks.json\').exists() else []\n    mapping={\'__DATA__\':b.to_json(orient=\'records\',force_ascii=False),\'__ROUNDS__\':r.to_json(orient=\'records\',force_ascii=False),\n             \'__PRED__\':p.to_json(orient=\'records\',force_ascii=False),\'__METRICS__\':json.dumps(m,ensure_ascii=False),\n             \'__QUALITY__\':json.dumps(q,ensure_ascii=False),\'__CHECKS__\':json.dumps(checks,ensure_ascii=False)}\n    page=(ROOT/\'src/dashboard_template.html\').read_text(encoding=\'utf-8\')\n    for key,value in mapping.items():page=page.replace(key,value.replace(\'</\',\'<\\\\/\'))\n    (ROOT/\'docs/index.html\').write_text(page,encoding=\'utf-8\')\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False,\'axes.titleweight\':\'bold\',\'axes.labelcolor\':\'#122235\',\'text.color\':\'#122235\'})\n    fig,ax=plt.subplots(figsize=(9.4,4.5));keys=[\'Unânime\',\'Dividida\'];v=[k[\'by_method\'][x][\'concordance\'] for x in keys]\n    ax.barh(keys[::-1],v[::-1],color=[\'#b63540\',\'#216aa0\'],height=.48);ax.set_xlim(0,100);ax.set_xlabel(\'Concordância (%) entre lutas sem empate em golpes\');ax.set_title(\'A contagem acompanha mais as decisões unânimes\',loc=\'left\',pad=18)\n    for i,key in enumerate(keys[::-1]):\n        a=k[\'by_method\'][key];ax.text(a[\'concordance\']+1,i,f"{a[\'concordance\']:.2f}%\\n{a[\'C\']}/{a[\'C\']+a[\'D\']}",va=\'center\',fontsize=11)\n    fig.tight_layout();fig.savefig(out/\'concordance.png\',dpi=180);plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9.4,4.3));ys=[2022,2023,2024]\n    for key,color in zip(keys,[\'#216aa0\',\'#b63540\']):\n        v=[k[\'by_year_method\'][f\'{y} {key}\'][\'concordance\'] for y in ys]\n        ax.plot(ys,v,\'o-\',label=key,color=color,linewidth=2.4)\n        for x,y in zip(ys,v):ax.annotate(f\'{y:.1f}%\',(x,y),xytext=(0,9),textcoords=\'offset points\',ha=\'center\',fontsize=10)\n    ax.set_ylim(0,100);ax.set_xticks(ys);ax.set_ylabel(\'Concordância (%)\');ax.set_title(\'A diferença mantém a mesma direção nos três anos\',loc=\'left\',pad=18);ax.legend(loc=\'lower right\');ax.grid(axis=\'y\',alpha=.15);fig.tight_layout();fig.savefig(out/\'years.png\',dpi=180);plt.close(fig)\n    g=r[r.bout_id==\'697efaf0d162e877\'];fig,ax=plt.subplots(figsize=(9.4,4.3));xx=np.arange(3)\n    for i,(name,color) in enumerate([(\'Kevin Holland\',\'#216aa0\'),(\'Jack Della Maddalena\',\'#b63540\')]):\n        d=g[g.fighter==name].sort_values(\'round\');pos=xx+(i-.5)*.36;bars=ax.bar(pos,d.sig_landed,width=.34,label=name,color=color);ax.bar_label(bars,padding=3)\n    ax.set_xticks(xx,[\'Round 1\',\'Round 2\',\'Round 3\']);ax.set_ylim(0,60);ax.set_ylabel(\'Golpes significativos conectados\');ax.set_title(\'Holland liderou 127 a 105; Della Maddalena venceu\',loc=\'left\',pad=18);ax.legend(loc=\'upper right\',fontsize=9);fig.tight_layout();fig.savefig(out/\'holland.png\',dpi=180);plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9.4,4.4));mat=np.array(m[\'test_model\'][\'confusion_matrix\']);ax.imshow(mat,cmap=\'Blues\',vmin=0,vmax=198)\n    for (i,j),v in np.ndenumerate(mat):ax.text(j,i,str(v),ha=\'center\',va=\'center\',fontsize=26,color=\'white\' if v>100 else \'#122235\')\n    ax.set_xticks([0,1],[\'Divergência\',\'Concordância\']);ax.set_yticks([0,1],[\'Divergência\',\'Concordância\']);ax.set_xlabel(\'Previsto\');ax.set_ylabel(\'Observado\');ax.set_title(\'25 das 49 divergências foram identificadas no teste\',pad=18);fig.tight_layout();fig.savefig(out/\'confusion.png\',dpi=180);plt.close(fig)\n    print(\'Dashboard e quatro gráficos gerados.\');return page\n\nif __name__==\'__main__\':build()\n', 'src/analytics.py': '"""Classificação retrospectiva do resultado de decisões, após três rounds.\n\nNão é uma previsão pré-luta. O modelo usa indicadores do combate concluído,\nmas nunca usa vencedor, método de decisão, votos ou placares como entrada.\nEscolha de C em 2023. Teste final em 2024, sem ajuste após observá-lo.\n"""\nimport json\nimport platform\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score,balanced_accuracy_score,roc_auc_score,log_loss,brier_score_loss,confusion_matrix,precision_score,recall_score,f1_score\nfrom .pipeline import ROOT\n\nFEATURES=[\'sig_margin\',\'attempt_margin\',\'accuracy_margin\',\'round_margin\',\'td_landed_margin\',\'kd_margin\',\'control_seconds_margin\']\n\ndef metrics(y,p):\n    predicted=(p>=0.5).astype(int)\n    return dict(n=len(y),accuracy=float(accuracy_score(y,predicted)),balanced_accuracy=float(balanced_accuracy_score(y,predicted)),\n                auc=float(roc_auc_score(y,p)),log_loss=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),\n                precision_concordance=float(precision_score(y,predicted,zero_division=0)),recall_concordance=float(recall_score(y,predicted,zero_division=0)),\n                f1_concordance=float(f1_score(y,predicted,zero_division=0)),recall_divergence=float(recall_score(y,predicted,pos_label=0,zero_division=0)),\n                confusion_matrix=confusion_matrix(y,predicted,labels=[0,1]).tolist())\n\ndef estimator(c):\n    return Pipeline([(\'imputer\',SimpleImputer(strategy=\'median\')),(\'scaler\',StandardScaler()),\n                     (\'model\',LogisticRegression(C=c,max_iter=3000,random_state=42))])\n\ndef run():\n    b=pd.read_csv(ROOT/\'data/processed/bouts.csv\');b=b[b.status.isin([\'C\',\'D\'])].copy()\n    b[\'target\']=(b.status==\'C\').astype(int)\n    train=b[b.year==2022];val=b[b.year==2023];test=b[b.year==2024]\n    assert train.date.max()<val.date.min()<test.date.min()\n    trials=[]\n    for c in (0.1,1.0,10.0):\n        model=estimator(c).fit(train[FEATURES],train.target)\n        trials.append({\'C\':c,**metrics(val.target,model.predict_proba(val[FEATURES])[:,1])})\n    chosen=min(trials,key=lambda x:(x[\'log_loss\'],x[\'C\']))[\'C\']\n    dev=pd.concat([train,val]).sort_values(\'date\')\n    model=estimator(chosen).fit(dev[FEATURES],dev.target)\n    probabilities=model.predict_proba(test[FEATURES])[:,1]\n    baseline=np.full(len(test),dev.target.mean())\n    result={\'task\':\'Probabilidade de o líder único em golpes significativos vencer a decisão, dados os três rounds completos.\',\n            \'prediction_time\':\'Após os três rounds; avaliação retrospectiva em decisões unânimes/divididas. Não é previsão pré-luta nem serviço ao vivo.\',\n            \'target\':\'1 = concordância; 0 = divergência\',\'features\':FEATURES,\'validation\':trials,\'selected_C\':chosen,\n            \'split\':{\'train_2022\':len(train),\'validation_2023\':len(val),\'final_train_2022_2023\':len(dev),\'test_2024\':len(test)},\n            \'test_model\':metrics(test.target,probabilities),\'test_baseline\':metrics(test.target,baseline),\n            \'baseline_probability\':float(dev.target.mean()),\'environment\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__}}\n    pred=test[[\'bout_id\',\'date\',\'method\',\'leader\',\'other\',\'winner\',\'status\',\'target\']].copy()\n    pred[\'probability\']=probabilities;pred[\'predicted\']=(probabilities>=0.5).astype(int);pred[\'baseline_probability\']=baseline\n    pred.to_csv(ROOT/\'outputs/predictions_2024.csv\',index=False)\n    result[\'test_by_method\']={str(m):metrics(g.target,g.probability.to_numpy()) for m,g in pred.groupby(\'method\')}\n    (ROOT/\'outputs/model_metrics.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    # JSON transparente evita desserialização de pickle. Permite reproduzir inferências.\n    artifact={\'features\':FEATURES,\'imputer\':model[\'imputer\'].statistics_.tolist(),\'mean\':model[\'scaler\'].mean_.tolist(),\n              \'scale\':model[\'scaler\'].scale_.tolist(),\'coef\':model[\'model\'].coef_[0].tolist(),\'intercept\':float(model[\'model\'].intercept_[0]),\n              \'training_last_date\':dev.date.max(),\'C\':chosen}\n    (ROOT/\'outputs/model.json\').write_text(json.dumps(artifact,indent=2),encoding=\'utf-8\')\n    logits=((test[FEATURES].fillna(dict(zip(FEATURES,artifact[\'imputer\'])))-artifact[\'mean\'])/artifact[\'scale\']).to_numpy()@np.array(artifact[\'coef\'])+artifact[\'intercept\']\n    assert np.allclose(1/(1+np.exp(-logits)),probabilities)\n    print(json.dumps(result,ensure_ascii=False,indent=2));return result\n\nif __name__==\'__main__\':run()\n', 'src/__init__.py': '"""Projeto UFC de BI e Analytics."""\n', 'src/dashboard_template.html': '<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>UFC | Golpes e decisões</title>\n<style>\n:root{--ink:#122235;--muted:#56667b;--line:#dce3eb;--paper:#f4f6f9;--blue:#216aa0;--red:#b63540;--green:#137f76;--gold:#a97515}*{box-sizing:border-box}body{margin:0;background:var(--paper);font:15px/1.5 system-ui,Arial;color:var(--ink)}header{background:#122235;color:white;padding:32px max(24px,calc((100% - 1192px)/2));border-top:5px solid #b63540}header small{letter-spacing:2px;color:#b9c9dc;font-weight:700}h1{font-size:34px;margin:8px 0 4px;line-height:1.2}header p{color:#d3deeb;margin:0;max-width:950px}main{max-width:1240px;margin:auto;padding:24px}nav{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:24px}button,select,input{font:inherit;border:1px solid #bdc9d7;border-radius:6px;padding:9px 12px;background:white;color:var(--ink)}button{cursor:pointer;font-weight:600}button:hover{border-color:var(--blue)}button:focus-visible,select:focus-visible,input:focus-visible{outline:3px solid #45a3d2;outline-offset:2px}nav button[aria-selected=true]{background:var(--ink);color:white;border-color:var(--ink)}.filters{display:flex;align-items:end;gap:16px;flex-wrap:wrap;margin-bottom:20px}label{display:flex;flex-direction:column;gap:5px;font-size:13px;font-weight:650}.grow{flex:1;min-width:200px}.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin:16px 0}.card,.panel{background:white;border:1px solid var(--line);border-radius:9px;padding:20px}.card small{color:var(--muted);font-size:13px}.value{font-size:31px;font-weight:750;line-height:1.3;margin:8px 0}.caption{font-size:13px;color:var(--muted)}h2{font-size:22px;margin:0 0 6px}h3{font-size:18px;margin:0 0 10px}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin:18px 0}.wide{grid-column:1/-1}.insight{font-size:17px;line-height:1.6;margin:20px 0;padding:0 2px}.insight strong{color:var(--blue)}svg{display:block;width:100%;height:auto;overflow:visible}.tablewrap{overflow:auto}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:11px 10px;text-align:left;border-bottom:1px solid var(--line)}th{background:#eef2f6;font-weight:700}tr:hover td{background:#f6f8fb}.pill{display:inline-block;border-radius:4px;background:#e4f0ef;padding:3px 8px;color:#0a625b;font-size:13px;font-weight:650}.pill.d{background:#f8e8e9;color:#922332}.pill.e{background:#faf0db;color:#76520c}a{color:#175b8c}p{margin:8px 0 14px}footer{max-width:1240px;margin:0 auto;padding:0 24px 30px;color:var(--muted);font-size:12px}.notice{border-left:3px solid #b63540;padding-left:14px;color:#384a60;margin:18px 0}.legend{display:flex;gap:18px;flex-wrap:wrap;font-size:13px;color:var(--muted)}.dot{display:inline-block;width:9px;height:9px;border-radius:100%;margin-right:5px}.hidden{display:none}.metric-table td:nth-child(n+2){text-align:right}.metric-table th:nth-child(n+2){text-align:right}.key{font-size:12px;letter-spacing:.5px;color:var(--muted);text-transform:uppercase}@media(max-width:760px){h1{font-size:26px}.cards{grid-template-columns:1fr 1fr}.grid{grid-template-columns:1fr}.value{font-size:26px}main{padding:16px}.card,.panel{padding:15px}header{padding:24px 16px}nav button{font-size:13px}.filters>label{flex:1}table{font-size:12px}}\n</style></head><body>\n<header><small>BI E ANALYTICS · PROJETO ACADÊMICO</small><h1>Quem lidera os golpes vence a decisão?</h1><p>UFC, 2022 a 2024 · Três rounds completos · Decisões unânimes e divididas · Igor de Andrade Albuquerque</p></header>\n<main><nav role="tablist" aria-label="Painéis"><button role="tab" aria-selected="true" data-page="overview">Panorama</button><button role="tab" aria-selected="false" data-page="bouts">Conferir uma luta</button><button role="tab" aria-selected="false" data-page="model">Modelo preditivo</button><button role="tab" aria-selected="false" data-page="quality">Dados e critérios</button></nav>\n<section id="overview"><div class="filters"><label>Ano<select id="year"><option value="">2022–2024</option><option>2022</option><option>2023</option><option>2024</option></select></label><label>Tipo de decisão<select id="method"><option value="">Todos</option><option>Unânime</option><option>Dividida</option></select></label><button id="reset">Limpar filtros</button><button id="export">Baixar lutas filtradas em CSV</button></div>\n<div class="cards" id="cards"></div><div id="headline" class="insight" aria-live="polite"></div>\n<div class="grid"><article class="panel"><h3>Concordância por tipo de decisão</h3><p class="caption">Percentual entre lutas com um líder único. Rótulos mostram acertos / total classificável.</p><div id="methods"></div></article><article class="panel"><h3>Concordância ao longo dos anos</h3><p class="caption">A escala começa em zero. O filtro de tipo de decisão também se aplica.</p><div id="years"></div></article><article class="panel wide"><h3>O que entrou em cada percentual</h3><div id="counts"></div></article></div>\n<p class="notice">Divergência na contagem não demonstra erro de arbitragem. A análise descreve associações no recorte e não substitui a avaliação round a round.</p></section>\n<section id="bouts" class="hidden"><h2>Dos indicadores aos três rounds</h2><p class="caption">Filtros próprios desta consulta. Selecione uma luta para conferir os registros que compõem o indicador.</p><div class="filters"><label class="grow">Buscar atleta ou evento<input id="search" placeholder="Ex.: Holland"></label><label>Classificação<select id="status"><option value="">Todas</option><option value="C">Concordância</option><option value="D">Divergência</option><option value="E">Empate em golpes</option></select></label></div><label>Luta<select id="fight"></select></label><div id="detail"></div><div class="panel"><h3>Lutas encontradas</h3><p class="caption" id="listcount"></p><div class="tablewrap" id="fighttable"></div></div></section>\n<section id="model" class="hidden"><h2>O modelo acrescenta informação à contagem?</h2><p>Classificação retrospectiva: estimar a probabilidade de o líder único vencer a decisão usando as estatísticas dos três rounds completos. O tipo de decisão e o vencedor não entram nas variáveis explicativas.</p><p class="notice"><strong>Momento da previsão:</strong> combate concluído. Não é previsão pré-luta. A disponibilidade em tempo real dessas estatísticas não foi validada.</p><div class="cards" id="modelcards"></div><div class="grid"><div class="panel"><h3>Comparação no teste de 2024</h3><div id="modeltable"></div><p class="caption">Baseline: probabilidade constante igual à taxa de concordância de 2022–2023; no limiar 0,5, prevê sempre vitória do líder.</p></div><div class="panel"><h3>Acertos e erros do modelo</h3><p class="caption">Matriz de confusão. Linhas = observado; colunas = previsto.</p><div id="matrix"></div><p id="modellimit" class="caption"></p></div></div><div class="panel"><h3>Reproduzir uma previsão fora do treino</h3><label>Luta reservada para teste em 2024<select id="prediction"></select></label><div id="predictiondetail" class="insight" aria-live="polite"></div><button id="exportpred">Baixar as 247 previsões em CSV</button></div><p class="caption">Treino inicial: 2022. Seleção de C pela menor log loss em 2023. Reajuste em 2022–2023 e teste final em 2024. O limiar permanece em 0,5. Os empates na contagem ficam fora deste classificador.</p></section>\n<section id="quality" class="hidden"><h2>Origem, regras e rastreabilidade</h2><div class="grid"><article class="panel"><h3>Fluxo reproduzível</h3><ol><li>Três CSVs do espelho Greco1899, com versão e SHA-256 fixados.</li><li>Reconciliação de nomes de eventos pela URL da luta.</li><li>Verificação de dois atletas, três rounds e acertos ≤ tentativas.</li><li>SQLite: luta, atleta e registros por round.</li><li>KPIs, teste temporal e painéis no notebook Colab.</li></ol></article><article class="panel"><h3>Qualidade observada</h3><div id="qualitynumbers"></div></article></div><div class="panel"><h3>Dicionário dos indicadores</h3><p><strong>C</strong>: líder venceu. <strong>D</strong>: líder perdeu. <strong>E</strong>: atletas empataram nos golpes significativos conectados.</p><p>Concordância = 100 × C / (C + D). Divergência = 100 × D / (C + D). Cobertura classificável = 100 × (C + D) / (C + D + E). Denominador zero = não calculável.</p><p>H1 compara a concordância global com 50%. H2 subtrai a taxa das divididas da taxa das unânimes. Sensibilidade aos empates: 100 × C / N até 100 × (C + E) / N. São limites de atribuição dos empates, não intervalos de confiança.</p><p id="sourcecheck"></p><p class="caption">Snapshot: <a href="https://github.com/Greco1899/scrape_ufc_stats/tree/c742842f29466758fedc3906dbad28d6d320a8da" target="_blank" rel="noopener">c742842f29466758fedc3906dbad28d6d320a8da</a>. Fonte primária: <a href="http://ufcstats.com/statistics/events/completed?page=all" target="_blank" rel="noopener">UFCStats</a>. O espelho é não oficial.</p></div></section>\n</main><footer>Projeto acadêmico independente, sem vínculo com o UFC. Resultados do snapshot especificado. Dados históricos de 2022–2024. Preparação e análise: 18/11/2026.</footer>\n<script>\nconst DATA=__DATA__, ROUNDS=__ROUNDS__, PRED=__PRED__, METRICS=__METRICS__, QUALITY=__QUALITY__, CHECKS=__CHECKS__;\nconst $=id=>document.getElementById(id);const fmt=(x,n=2)=>x===null||!Number.isFinite(x)?\'Não calculável\':x.toLocaleString(\'pt-BR\',{minimumFractionDigits:n,maximumFractionDigits:n});const esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nfunction stats(rows){let C=0,D=0,E=0;for(const r of rows){if(r.status===\'C\')C++;else if(r.status===\'D\')D++;else E++}const n=C+D+E;return {n,C,D,E,p:C+D?100*C/(C+D):null,coverage:n?100*(C+D)/n:null}}\nfunction card(label,value,caption){return `<article class="card"><small>${label}</small><div class="value">${value}</div><div class="caption">${caption}</div></article>`}\nfunction bars(items,max=100){let h=items.length*72+30;return `<svg role="img" aria-label="Gráfico de barras" viewBox="0 0 530 ${h}">${items.map((d,i)=>{let y=10+i*72;return `<text x="0" y="${y+17}" fill="#122235" font-size="14">${esc(d.label)}</text><rect x="0" y="${y+28}" width="370" height="18" rx="3" fill="#edf1f5"/><rect x="0" y="${y+28}" width="${d.value===null?0:Math.max(0,d.value)/max*370}" height="18" rx="3" fill="${d.color||\'#216aa0\'}"/><text x="384" y="${y+43}" font-size="16" fill="#122235" font-weight="700">${d.value===null?\'N/C\':fmt(d.value)+\'%\'}</text><text x="0" y="${y+64}" fill="#56667b" font-size="12">${esc(d.note||\'\')}</text>`}).join(\'\')}</svg>`}\nfunction table(headers,rows,cls=\'\'){return `<table class="${cls}"><thead><tr>${headers.map(h=>\'<th scope="col">\'+h+\'</th>\').join(\'\')}</tr></thead><tbody>${rows.map(r=>\'<tr>\'+r.map(x=>\'<td>\'+x+\'</td>\').join(\'\')+\'</tr>\').join(\'\')}</tbody></table>`}\nfunction getFiltered(){return DATA.filter(r=>(!$(\'year\').value||r.year===+$(\'year\').value)&&(!$(\'method\').value||r.method===$(\'method\').value))}\nfunction renderOverview(){const rows=getFiltered(),k=stats(rows);$(\'cards\').innerHTML=card(\'Lutas válidas\',fmt(k.n,0),\'C + D + E no filtro\')+card(\'Concordância\',k.p===null?\'N/C\':fmt(k.p)+\'%\',`${k.C} / ${k.C+k.D} lutas com líder`)+card(\'Divergências\',fmt(k.D,0),`${fmt(k.C+k.D?100*k.D/(k.C+k.D):null)}% das classificáveis`)+card(\'Empates em golpes\',fmt(k.E,0),`Cobertura classificável: ${fmt(k.coverage)}%`);\nconst u=stats(rows.filter(r=>r.method===\'Unânime\')),d=stats(rows.filter(r=>r.method===\'Dividida\'));$(\'headline\').innerHTML=k.n?`O líder venceu em <strong>${k.C} de ${k.C+k.D}</strong> lutas classificáveis.${u.p!==null&&d.p!==null?` Nas unânimes, a concordância foi <strong>${fmt(u.p-d.p)} pontos percentuais maior</strong> que nas divididas.`:\'\'}`:\'Nenhuma luta encontrada para estes filtros.\';\n$(\'methods\').innerHTML=bars([{label:\'Unânime\',value:u.p,note:`${u.C} / ${u.C+u.D} · ${u.E} empate(s)`,color:\'#216aa0\'},{label:\'Dividida\',value:d.p,note:`${d.C} / ${d.C+d.D} · ${d.E} empate(s)`,color:\'#b63540\'}]);$(\'years\').innerHTML=bars([2022,2023,2024].filter(y=>!$(\'year\').value||y===+$(\'year\').value).map(y=>{let a=stats(rows.filter(r=>r.year===y));return {label:String(y),value:a.p,note:`${a.C} / ${a.C+a.D} · ${a.E} empate(s)`}}));\n$(\'counts\').innerHTML=table([\'Classificação\',\'Lutas\',\'Participação no total válido\'],[[\'Concordância\',k.C,fmt(k.n?100*k.C/k.n:null)+\'%\'],[\'Divergência\',k.D,fmt(k.n?100*k.D/k.n:null)+\'%\'],[\'Empate na contagem\',k.E,fmt(k.n?100*k.E/k.n:null)+\'%\']]);}\nfunction download(rows,name){if(!rows.length)return;let cols=Object.keys(rows[0]),csv=\'\\ufeff\'+[cols,...rows.map(r=>cols.map(c=>r[c]??\'\'))].map(a=>a.map(x=>\'"\'+String(x).replaceAll(\'"\',\'""\')+\'"\').join(\';\')).join(\'\\r\\n\');let a=document.createElement(\'a\');a.href=URL.createObjectURL(new Blob([csv],{type:\'text/csv;charset=utf-8;\'}));a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000)}\nfunction renderList(prefer){let q=$(\'search\').value.toLocaleLowerCase(\'pt-BR\'),rows=DATA.filter(r=>(!$(\'status\').value||r.status===$(\'status\').value)&&`${r.fighter_a} ${r.fighter_b} ${r.event}`.toLocaleLowerCase(\'pt-BR\').includes(q)).sort((a,b)=>b.date.localeCompare(a.date));let prior=prefer||$(\'fight\').value;$(\'fight\').innerHTML=rows.map(r=>`<option value="${r.bout_id}">${r.date} · ${esc(r.fighter_a)} × ${esc(r.fighter_b)}</option>`).join(\'\');if(rows.some(r=>r.bout_id===prior))$(\'fight\').value=prior;$(\'listcount\').textContent=`${rows.length} luta(s). Tabela mostra as primeiras ${Math.min(50,rows.length)}; a seleção acima inclui todas.`;$(\'fighttable\').innerHTML=table([\'Data\',\'Confronto\',\'Decisão\',\'Classificação\'],rows.slice(0,50).map(r=>[r.date,esc(r.fighter_a)+\' × \'+esc(r.fighter_b),esc(r.method),label(r.status)]));renderDetail()}\nfunction label(s){return s===\'C\'?\'Concordância\':s===\'D\'?\'Divergência\':\'Empate em golpes\'}\nfunction renderDetail(){let r=DATA.find(r=>r.bout_id===$(\'fight\').value);if(!r){$(\'detail\').innerHTML=\'<p>Nenhuma luta encontrada.</p>\';return}let rounds=ROUNDS.filter(x=>x.bout_id===r.bout_id),max=Math.max(...rounds.map(x=>x.sig_landed),1),plot=\'\';for(let n=1;n<=3;n++){let a=rounds.find(x=>x.fighter===r.fighter_a&&x.round===n),b=rounds.find(x=>x.fighter===r.fighter_b&&x.round===n),y=(n-1)*100+20;plot+=`<text x="0" y="${y+20}" font-size="14">Round ${n}</text>`;for(let [i,x] of [a,b].entries()){let w=x.sig_landed/max*340,yy=y+i*30;plot+=`<rect x="100" y="${yy}" width="${w}" height="21" rx="2" fill="${i?\'#b63540\':\'#216aa0\'}"/><text x="${110+w}" y="${yy+16}" font-size="14">${x.sig_landed} / ${x.sig_attempted}</text>`}}\n$(\'detail\').innerHTML=`<div class="grid"><article class="panel"><span class="pill ${r.status.toLowerCase()}">${label(r.status)}</span><h3 style="margin-top:14px">${esc(r.fighter_a)} × ${esc(r.fighter_b)}</h3><p><strong>Vencedor oficial:</strong> ${esc(r.winner)}<br>Decisão ${esc(r.method.toLowerCase())} · ${r.date}</p><p>Acertos totais: <strong>${r.sig_a} × ${r.sig_b}</strong></p><p class="caption">${esc(r.event)}<br>${esc(r.weightclass)}</p><a href="${esc(r.url)}" target="_blank" rel="noopener">Abrir ficha no UFCStats</a></article><article class="panel"><h3>Golpes significativos por round</h3><div class="legend"><span><i class="dot" style="background:#216aa0"></i>${esc(r.fighter_a)}</span><span><i class="dot" style="background:#b63540"></i>${esc(r.fighter_b)}</span></div><svg role="img" aria-label="Acertos por round" viewBox="0 0 560 330">${plot}</svg><p class="caption">Rótulos: acertos / tentativas. Barras representam somente acertos, na mesma escala.</p></article></div>`}\nfunction renderModel(){let m=METRICS.test_model,b=METRICS.test_baseline;$(\'modelcards\').innerHTML=card(\'Teste sem empates\',m.n,\'Lutas de 2024\')+card(\'Acurácia do modelo\',fmt(100*m.accuracy)+\'%\',`${m.confusion_matrix[0][0]+m.confusion_matrix[1][1]} previsões corretas`)+card(\'Ganho sobre baseline\',fmt(100*(m.accuracy-b.accuracy))+\' p.p.\',\'Comparação no mesmo teste\')+card(\'Recall das divergências\',fmt(100*m.recall_divergence)+\'%\',\'25 das 49 divergências identificadas\');$(\'modeltable\').innerHTML=table([\'Métrica\',\'Baseline\',\'Modelo\'],[[\'Acurácia\',fmt(b.accuracy*100)+\'%\',fmt(m.accuracy*100)+\'%\'],[\'Acurácia balanceada\',fmt(b.balanced_accuracy*100)+\'%\',fmt(m.balanced_accuracy*100)+\'%\'],[\'ROC-AUC\',fmt(b.auc,3),fmt(m.auc,3)],[\'Log loss ↓\',fmt(b.log_loss,3),fmt(m.log_loss,3)],[\'Brier ↓\',fmt(b.brier,3),fmt(m.brier,3)]],\'metric-table\');let a=m.confusion_matrix;$(\'matrix\').innerHTML=table([\'Observado / previsto\',\'Divergência\',\'Concordância\'],[[\'Divergência\',a[0][0],a[0][1]],[\'Concordância\',a[1][0],a[1][1]]],\'metric-table\');$(\'modellimit\').textContent=\'Nas 59 decisões divididas do teste, a acurácia foi 67,80%, igual à regra de sempre escolher o líder. O ganho agregado concentra-se nas decisões unânimes.\';$(\'prediction\').innerHTML=PRED.map(r=>`<option value="${r.bout_id}">${r.date} · ${esc(r.leader)} × ${esc(r.other)}</option>`).join(\'\');renderPrediction()}\nfunction renderPrediction(){const r=PRED.find(x=>x.bout_id===$(\'prediction\').value);if(!r)return;$(\'predictiondetail\').innerHTML=`Probabilidade estimada de vitória do líder <strong>${esc(r.leader)}: ${fmt(100*r.probability)}%</strong>.<br>Previsão no limiar 0,5: ${r.predicted?\'concordância\':\'divergência\'}. Resultado observado: ${label(r.status).toLowerCase()} (${esc(r.winner)}).`}\n$(\'qualitynumbers\').innerHTML=table([\'Verificação\',\'Resultado\'],[[\'Lutas elegíveis / válidas\',`${QUALITY.eligible_bouts} / ${QUALITY.valid_bouts}`],[\'Linhas por round antes / depois\',`${QUALITY.round_rows_before_dedup} / ${QUALITY.round_rows_after_dedup}`],[\'Lutas excluídas por falha no recorte\',QUALITY.excluded_bouts],[\'Campos opcionais ausentes no recorte\',Object.values(QUALITY.optional_missing).reduce((a,b)=>a+b,0)],[\'Registros sem data no arquivo completo\',QUALITY.result_rows_without_date_after_alias]]);let verified=CHECKS.filter(x=>x.field_validation===\'verificada\').length;$(\'sourcecheck\').textContent=`Conferência externa: ${verified} de ${CHECKS.length} fichas com campos validados nesta execução. O acesso às fichas oficiais ficou indisponível; a reconciliação interna não substitui essa verificação. Os registros e URLs estão em outputs/source_checks.json.`;\nfor(const button of document.querySelectorAll(\'nav button\'))button.onclick=()=>{for(const b of document.querySelectorAll(\'nav button\'))b.setAttribute(\'aria-selected\',String(b===button));for(const id of [\'overview\',\'bouts\',\'model\',\'quality\'])$(id).classList.toggle(\'hidden\',id!==button.dataset.page)};\n$(\'year\').onchange=$(\'method\').onchange=renderOverview;$(\'reset\').onclick=()=>{$(\'year\').value=\'\';$(\'method\').value=\'\';renderOverview()};$(\'export\').onclick=()=>download(getFiltered(),\'ufc_lutas_filtradas.csv\');$(\'exportpred\').onclick=()=>download(PRED,\'ufc_previsoes_2024.csv\');$(\'search\').oninput=()=>renderList();$(\'status\').onchange=()=>renderList();$(\'fight\').onchange=renderDetail;$(\'prediction\').onchange=renderPrediction;renderOverview();renderList(\'697efaf0d162e877\');renderModel();\n</script></body></html>\n', 'tests/test_project.py': '"""Casos limites e reconciliação independente dos artefatos calculados."""\nimport json\nimport sqlite3\nimport unittest\nimport pandas as pd\nfrom src.pipeline import ROOT,pair,outcome_winner,classify,kpis\nfrom src.analytics import FEATURES\n\nclass EdgeCases(unittest.TestCase):\n    def test_pair_valid(self):self.assertEqual(pair(\'12 of 30\'),(12,30))\n    def test_zero_is_valid(self):self.assertEqual(pair(\'0 of 0\'),(0,0))\n    def test_missing_is_not_zero(self):\n        with self.assertRaises(ValueError):pair(\'\')\n    def test_landed_cannot_exceed_attempted(self):\n        with self.assertRaises(ValueError):pair(\'31 of 30\')\n    def test_negative_is_invalid(self):\n        with self.assertRaises(ValueError):pair(\'-1 of 30\')\n    def test_outcome_orientation(self):\n        self.assertEqual(outcome_winner(\'A vs. B\',\'W/L\'),\'A\');self.assertEqual(outcome_winner(\'A vs. B\',\'L/W\'),\'B\')\n    def test_tie_and_divergence(self):\n        a={\'fighter\':\'A\',\'sig_landed\':10};b={\'fighter\':\'B\',\'sig_landed\':10}\n        self.assertEqual(classify(a,b,\'A\'),\'E\');a[\'sig_landed\']=11\n        self.assertEqual(classify(a,b,\'B\'),\'D\');self.assertEqual(classify(a,b,\'A\'),\'C\')\n    def test_empty_filter(self):self.assertIsNone(kpis(pd.DataFrame({\'status\':[]}))[\'concordance\'])\n    def test_only_ties(self):\n        result=kpis(pd.DataFrame({\'status\':[\'E\',\'E\']}));self.assertIsNone(result[\'concordance\']);self.assertEqual(result[\'coverage\'],0)\n\nclass Integration(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.b=pd.read_csv(ROOT/\'data/processed/bouts.csv\');cls.r=pd.read_csv(ROOT/\'data/processed/rounds.csv\')\n    def test_grain(self):\n        self.assertFalse(self.b.bout_id.duplicated().any());self.assertFalse(self.r.duplicated([\'bout_id\',\'fighter\',\'round\']).any())\n        self.assertTrue(self.r.groupby(\'bout_id\').size().eq(6).all())\n    def test_round_totals(self):\n        total=self.r.groupby([\'bout_id\',\'fighter\']).sig_landed.sum()\n        for x in self.b.itertuples():\n            self.assertEqual(total.loc[x.bout_id,x.fighter_a],x.sig_a);self.assertEqual(total.loc[x.bout_id,x.fighter_b],x.sig_b)\n    def test_count_reconciliation_all_filters(self):\n        for year in [None,2022,2023,2024]:\n            for method in [None,\'Unânime\',\'Dividida\']:\n                f=self.b.copy()\n                if year:f=f[f.year==year]\n                if method:f=f[f.method==method]\n                k=kpis(f);self.assertEqual(k[\'C\']+k[\'D\']+k[\'E\'],len(f))\n    def test_sql_matches_python(self):\n        with sqlite3.connect(ROOT/\'data/processed/ufc.sqlite\') as c:s=pd.read_sql(\'SELECT * FROM vw_bout_kpis\',c)\n        for x in s.itertuples():\n            k=kpis(self.b[(self.b.year==x.year)&(self.b.method==x.method)])\n            self.assertAlmostEqual(k[\'concordance\'],x.concordance_pct)\n    def test_holland_example(self):\n        row=self.b[self.b.bout_id==\'697efaf0d162e877\'].iloc[0]\n        self.assertEqual(row.winner,\'Jack Della Maddalena\');self.assertEqual(row.status,\'D\');self.assertEqual({row.sig_a,row.sig_b},{127,105})\n    def test_no_result_feature(self):\n        self.assertFalse(set(FEATURES)&{\'winner\',\'method\',\'status\',\'target\',\'OUTCOME\',\'DETAILS\',\'date\',\'year\'})\n    def test_test_set_is_temporal_and_unique(self):\n        p=pd.read_csv(ROOT/\'outputs/predictions_2024.csv\');self.assertTrue(p.date.str.startswith(\'2024\').all());self.assertFalse(p.bout_id.duplicated().any())\n        train=self.b[self.b.year<2024];self.assertFalse(set(train.bout_id)&set(p.bout_id))\n    def test_model_matrix(self):\n        p=pd.read_csv(ROOT/\'outputs/predictions_2024.csv\');m=json.loads((ROOT/\'outputs/model_metrics.json\').read_text())\n        self.assertEqual(int((p.predicted==p.target).sum()),sum(m[\'test_model\'][\'confusion_matrix\'][i][i] for i in [0,1]))\n\nif __name__==\'__main__\':unittest.main()\n', 'outputs/source_checks.json': '[\n  {\n    "bout_id": "0366e183296fea07",\n    "url": "http://ufcstats.com/fight-details/0366e183296fea07",\n    "checked_at": "2026-11-18T19:38:31.098162+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "04e2d50c24a88dbd",\n    "url": "http://ufcstats.com/fight-details/04e2d50c24a88dbd",\n    "checked_at": "2026-11-18T19:38:31.099014+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "029f5c36820fdc09",\n    "url": "http://ufcstats.com/fight-details/029f5c36820fdc09",\n    "checked_at": "2026-11-18T19:38:31.099628+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "0489b42ac0ab6d4b",\n    "url": "http://ufcstats.com/fight-details/0489b42ac0ab6d4b",\n    "checked_at": "2026-11-18T19:38:31.100258+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "06d9fb9c995f1e4f",\n    "url": "http://ufcstats.com/fight-details/06d9fb9c995f1e4f",\n    "checked_at": "2026-11-18T19:38:38.931944+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "0732e6b6eb0a607d",\n    "url": "http://ufcstats.com/fight-details/0732e6b6eb0a607d",\n    "checked_at": "2026-11-18T19:38:38.942794+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "009feda296ae589c",\n    "url": "http://ufcstats.com/fight-details/009feda296ae589c",\n    "checked_at": "2026-11-18T19:38:38.950231+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "0197a61472fc351d",\n    "url": "http://ufcstats.com/fight-details/0197a61472fc351d",\n    "checked_at": "2026-11-18T19:38:38.952332+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "01f7c28804746979",\n    "url": "http://ufcstats.com/fight-details/01f7c28804746979",\n    "checked_at": "2026-11-18T19:38:44.463845+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "059f5cf0e9e37627",\n    "url": "http://ufcstats.com/fight-details/059f5cf0e9e37627",\n    "checked_at": "2026-11-18T19:38:44.487668+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "01432891260084a4",\n    "url": "http://ufcstats.com/fight-details/01432891260084a4",\n    "checked_at": "2026-11-18T19:38:44.496513+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  },\n  {\n    "bout_id": "02ec66f311dce539",\n    "url": "http://ufcstats.com/fight-details/02ec66f311dce539",\n    "checked_at": "2026-11-18T19:38:44.513715+00:00",\n    "field_validation": "pendente",\n    "http_status": 200,\n    "access": "bloqueio de acesso; resposta sem tabela da luta",\n    "error": "HTTP 200 trouxe página de verificação de navegador, sem estatísticas."\n  }\n]'}
for relative, text in SOURCES.items():
    (ROOT / relative).write_text(text, encoding='utf-8')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Ambiente pronto. Diretório de trabalho:', ROOT)


Ambiente pronto. Diretório de trabalho: /workspace/scratch/19f031bb90fc/output/Fase2_UFC/repositorio


## 2. Coleta, limpeza, armazenamento e KPIs
O download fixa um commit e valida SHA-256. Aliases são conciliados pela URL. Cada luta exige dois atletas × três rounds. Não há imputação de golpes significativos. Os arquivos tratados e o banco SQLite são gravados em `data/processed/`.

In [2]:
from src.pipeline import build
bouts, rounds = build()

{
  "quality": {
    "raw_rows": {
      "ufc_event_details.csv": 796,
      "ufc_fight_results.csv": 9015,
      "ufc_fight_stats.csv": 42356
    },
    "result_rows_without_date_before_alias": 27,
    "result_rows_without_date_after_alias": 2,
    "bouts_in_period": 1548,
    "eligible_result_rows": 669,
    "eligible_bouts": 665,
    "round_rows_before_dedup": 4014,
    "round_rows_after_dedup": 3990,
    "valid_bouts": 665,
    "excluded_bouts": 0,
    "valid_round_rows": 3990,
    "optional_missing": {
      "bout_id": 0,
      "fighter": 0,
      "round": 0,
      "sig_landed": 0,
      "sig_attempted": 0,
      "td_landed": 0,
      "kd": 0,
      "control_seconds": 0
    }
  },
  "kpis": {
    "all": {
      "n": 665,
      "C": 514,
      "D": 146,
      "E": 5,
      "concordance": 77.87878787878788,
      "divergence": 22.12121212121212,
      "coverage": 99.24812030075188,
      "sensitivity_low": 77.29323308270676,
      "sensitivity_high": 78.04511278195488
    },
    "by

## 3. Análise preditiva com teste temporal
Regressão logística: P(líder vencer | estatísticas dos três rounds). Sete diferenças líder–adversário: acertos, tentativas, precisão, número de rounds liderados, quedas, knockdowns e segundos de controle.

Treino: 2022. Validação: 2023, com C ∈ {0,1; 1; 10}, escolhido por log loss. Reajuste em 2022–2023. Teste final: 2024. Transformações são ajustadas exclusivamente nos dados de treino. Vencedor, placares e tipo de decisão não são atributos do modelo. Não reajuste o modelo após olhar o teste para apresentar uma estimativa honesta de desempenho.

In [3]:
from src.analytics import run
metrics = run()

{
  "task": "Probabilidade de o líder único em golpes significativos vencer a decisão, dados os três rounds completos.",
  "prediction_time": "Após os três rounds; avaliação retrospectiva em decisões unânimes/divididas. Não é previsão pré-luta nem serviço ao vivo.",
  "target": "1 = concordância; 0 = divergência",
  "features": [
    "sig_margin",
    "attempt_margin",
    "accuracy_margin",
    "round_margin",
    "td_landed_margin",
    "kd_margin",
    "control_seconds_margin"
  ],
  "validation": [
    {
      "C": 0.1,
      "n": 209,
      "accuracy": 0.8038277511961722,
      "balanced_accuracy": 0.6167644705254735,
      "auc": 0.8464923979727927,
      "log_loss": 0.3850351589557614,
      "brier": 0.12376708442024184,
      "precision_concordance": 0.824468085106383,
      "recall_concordance": 0.950920245398773,
      "f1_concordance": 0.8831908831908832,
      "recall_divergence": 0.2826086956521739,
      "confusion_matrix": [
        [
          13,
          33
        ]

## 4. Resultados visuais e interpretação
Os gráficos abaixo vêm da mesma base dos indicadores e do relatório. Um percentual maior no período não demonstra causalidade. A separação entre os tipos de decisão é descritiva.

In [4]:
from src.visuals import build as build_visuals
from IPython.display import display, Image, HTML
page = build_visuals()
for name in ['concordance','years','holland','confusion']:
    display(Image(filename=str(ROOT / f'outputs/{name}.png'), width=850))

Dashboard e quatro gráficos gerados.


In [5]:
kpis = json.loads((ROOT/'outputs/kpis.json').read_text())
k = kpis['all']
print(f"Concordância: {k['C']}/{k['C']+k['D']} = {k['concordance']:.2f}%. Empates: {k['E']}.")
print(f"H1: {kpis['H1_excess_pp']:.2f} p.p. acima de 50%. H2: diferença de {kpis['H2_difference_pp']:.2f} p.p.")
print('As duas hipóteses recebem apoio descritivo no recorte. Não são testes de erro de arbitragem.')
print('No teste de 2024, o classificador melhora o resultado agregado, mas sua acurácia nas decisões divididas não supera a regra de escolher sempre o líder.')

Concordância: 514/660 = 77.88%. Empates: 5.
H1: 27.88 p.p. acima de 50%. H2: diferença de 18.47 p.p.
As duas hipóteses recebem apoio descritivo no recorte. Não são testes de erro de arbitragem.
No teste de 2024, o classificador melhora o resultado agregado, mas sua acurácia nas decisões divididas não supera a regra de escolher sempre o líder.


## 5. Painéis interativos
As abas permitem consultar os KPIs, filtrar ano e tipo de decisão, conferir cada luta por round e reproduzir previsões de 2024. A aba do modelo tem seu próprio recorte fixo de teste. O arquivo `docs/index.html` também abre diretamente no navegador, sem servidor. No GitHub, abra este notebook no Colab e execute esta célula para usar as interações.

In [6]:
display(HTML(page))

## 6. Testes e reconciliação
Inclui orientação W/L e L/W, valores ausentes, empate, filtro vazio, grade de rounds, igualdade entre SQL e Python e separação temporal.

In [7]:
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'))
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_empty_filter (test_project.EdgeCases.test_empty_filter) ... ok
test_landed_cannot_exceed_attempted (test_project.EdgeCases.test_landed_cannot_exceed_attempted) ... ok
test_missing_is_not_zero (test_project.EdgeCases.test_missing_is_not_zero) ... ok
test_negative_is_invalid (test_project.EdgeCases.test_negative_is_invalid) ... ok
test_only_ties (test_project.EdgeCases.test_only_ties) ... ok
test_outcome_orientation (test_project.EdgeCases.test_outcome_orientation) ... ok
test_pair_valid (test_project.EdgeCases.test_pair_valid) ... ok
test_tie_and_divergence (test_project.EdgeCases.test_tie_and_divergence) ... ok
test_zero_is_valid (test_project.EdgeCases.test_zero_is_valid) ... ok
test_count_reconciliation_all_filters (test_project.Integration.test_count_reconciliation_all_filters) ... ok
test_grain (test_project.Integration.test_grain) ... ok
test_holland_example (test_project.Integration.test_holland_example) ... ok
test_model_matrix (test_project.Integration.test_model_matrix) .

## 7. Conferência das fontes e exportações
`outputs/source_checks.json` registra a tentativa de acesso às 12 fichas em 18/11/2026. Resposta HTTP 200 com verificação de navegador não foi contada como validação. Para repetir a coleta dessas páginas, execute `python -m src.source_check` e confronte os campos com os CSVs.

Artefatos: `data/processed/ufc.sqlite`, `bouts.csv`, `rounds.csv`, `outputs/predictions_2024.csv`, `model_metrics.json`, `model.json`, `kpis.json`, `quality.json` e `docs/index.html`.

Fontes: [snapshot Greco1899](https://github.com/Greco1899/scrape_ufc_stats/tree/c742842f29466758fedc3906dbad28d6d320a8da), [UFCStats](http://ufcstats.com/statistics/events/completed?page=all), [Colab](https://research.google.com/colaboratory/faq.html), [scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html). Metodologia e avaliação seguem as Aulas 7 e 10. Os conceitos de indicadores, filtros e apresentação articulam as Aulas 6 e 9.